### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
cd Your_Dir/emg2qwerty

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: Link the dataset

The training config expects data at `data/` inside the project folder. Since symlinks don't survive on Google Drive, we recreate it here. Adjust the source path if your data folder has a different name.

In [ ]:
# Create symlink: data/ -> emg2qwerty_data_one_user/
# (Symlinks from local don't transfer to Google Drive, so we recreate it here)
!ln -sfn emg2qwerty_data_one_user data
!ls data/*.hdf5 | head -5  # verify files are accessible

### Step 5: Start your experiments!

- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### (Optional) Launch TensorBoard for live training graphs

Run this cell **before** training to see live loss/metric curves. Click the refresh button in the TensorBoard panel to update during training.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

#### Training (Baseline TDS Conv)

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [ ]:
# Single-user training (baseline TDS Conv model)
!python -m emg2qwerty.train \
  user="single_user" \
  model=tds_conv_ctc \
  trainer.accelerator=gpu trainer.devices=1

#### Training (LSTM Model)

- Uses LSTM encoder with 2 fully connected layers instead of TDS Conv.
- Checkpoints saved in the same `logs` folder.

In [1]:
# Single-user training (LSTM model)
!python -m emg2qwerty.train \
  user="single_user" \
  model=lstm_ctc \
  trainer.accelerator=gpu trainer.devices=1

[2026-03-05 00:34:48,954][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

#### Testing

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.
- Make sure to use the matching `model=` config for the checkpoint you're testing.

In [ ]:
# Single-user testing (baseline TDS Conv)
!python -m emg2qwerty.train \
  user="single_user" \
  model=tds_conv_ctc \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

In [ ]:
# Single-user testing (LSTM model)
!python -m emg2qwerty.train \
  user="single_user" \
  model=lstm_ctc \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun